# 37. SegFormer MLP Decoder 구조

이 노트북은 `36_Efficient_Self_Attention_이해.ipynb` 다음 단계로, SegFormer decoder가 multi-scale feature를 segmentation map으로 바꾸는 과정을 이해합니다.

SegFormer decoder는 복잡한 decoder block을 많이 쌓기보다, 각 stage feature를 MLP로 같은 channel dimension에 맞추고 같은 해상도로 upsample한 뒤 concat합니다. 이후 간단한 prediction layer로 class logits를 만듭니다.

이번 노트북의 목표는 다음과 같습니다.

- MLP decoder의 입력과 출력 shape을 이해합니다.
- multi-scale feature를 같은 해상도와 channel로 맞추는 과정을 확인합니다.
- concat 후 segmentation logits를 만드는 흐름을 파악합니다.
- 단순한 decoder가 가지는 장점과 trade-off를 정리합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 37-1. Decoder 입력 feature

MiT encoder는 네 단계 feature를 출력합니다. Decoder는 이 feature들을 모두 활용해 segmentation logits를 만듭니다.

In [ ]:
features = [
    np.random.normal(size=(56, 56, 64)),
    np.random.normal(size=(28, 28, 128)),
    np.random.normal(size=(14, 14, 320)),
    np.random.normal(size=(7, 7, 512)),
]
for i, f in enumerate(features, start=1):
    print(f'C{i}: {f.shape}')

## 37-2. MLP projection

각 feature는 channel 수가 다릅니다. Decoder는 각 stage feature를 같은 embedding dimension으로 projection합니다. 실제 구현에서는 `Linear` 또는 `1x1 convolution`처럼 각 위치의 channel을 바꾸는 연산으로 볼 수 있습니다.

In [ ]:
decoder_dim = 32

def project_feature(feature, out_dim):
    h, w, c = feature.shape
    weight = np.random.normal(scale=0.1, size=(c, out_dim))
    return feature.reshape(-1, c) @ weight

projected = []
for f in features:
    h, w, c = f.shape
    p = project_feature(f, decoder_dim).reshape(h, w, decoder_dim)
    projected.append(p)

for i, p in enumerate(projected, start=1):
    print(f'projected C{i}: {p.shape}')

## 37-3. 같은 해상도로 upsample

SegFormer decoder는 모든 stage feature를 가장 높은 feature 해상도인 `H/4 x W/4`에 맞춥니다. 여기서는 nearest upsample로 단순화합니다.

In [ ]:
target_h, target_w = projected[0].shape[:2]

def upsample_to(feature, target_h, target_w):
    h, w, c = feature.shape
    scale_h = target_h // h
    scale_w = target_w // w
    return np.repeat(np.repeat(feature, scale_h, axis=0), scale_w, axis=1)

upsampled = [upsample_to(p, target_h, target_w) for p in projected]
for i, u in enumerate(upsampled, start=1):
    print(f'upsampled C{i}: {u.shape}')

In [ ]:
concat_feature = np.concatenate(upsampled, axis=-1)
num_classes = 3
classifier = np.random.normal(scale=0.05, size=(concat_feature.shape[-1], num_classes))
logits = concat_feature.reshape(-1, concat_feature.shape[-1]) @ classifier
logits = logits.reshape(target_h, target_w, num_classes)
pred = logits.argmax(axis=-1)

print('concat feature:', concat_feature.shape)
print('segmentation logits:', logits.shape)
print('prediction map:', pred.shape)

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(pred, cmap='tab10')
plt.title('예시 decoder prediction map')
plt.axis('off')
plt.show()

## 37-4. MLP decoder의 의미

SegFormer decoder는 복잡한 구조가 아닙니다. 중요한 계산은 encoder에서 수행하고, decoder는 multi-scale feature를 정렬하고 결합하는 역할에 집중합니다.

장점은 다음과 같습니다.

- decoder가 가벼워 추론 비용이 낮습니다.
- 서로 다른 stage feature를 단순한 방식으로 결합합니다.
- segmentation head를 이해하고 수정하기 쉽습니다.

trade-off는 decoder가 매우 정교한 boundary refinement를 직접 수행하지는 않는다는 점입니다. 경계 품질은 encoder feature와 학습 데이터, loss 설계에도 크게 의존합니다.

## 정리

- SegFormer decoder는 stage별 feature를 같은 channel dimension으로 projection합니다.
- 모든 feature를 `H/4 x W/4` 해상도로 upsample한 뒤 concat합니다.
- concat feature에서 class logits를 만들고 최종 segmentation map을 얻습니다.
- 다음 노트북 `38_SegFormer_전체_Forward_흐름.ipynb`에서는 encoder와 decoder를 하나의 forward 흐름으로 연결합니다.